# Human-in-the-Loop

Some jobs are too important for an agent to do alone, like promoting a ninja to ANBU. With
`needs_approval`, the run stops and waits for a human. The human says yes or no using a
`RunState`. Then the run starts again, even in a different program.


## Pause a tool call with `needs_approval`

`@function_tool(needs_approval=True)` means the tool never runs by itself. The run
stops, and the waiting call shows up in `RunResult.interruptions`.


In [1]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: the action every demo here pauses, approves, or rejects.
# Wired onto anbu_review_agent's tools=[promote_to_anbu] below
@function_tool(needs_approval=True)  # needs_approval: pauses the run instead of running the tool
def promote_to_anbu(ninja_name: str) -> str:
    """Promote a ninja to ANBU. Requires human approval."""
    return f"{ninja_name} has been promoted to ANBU."

In [2]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)

# Initialize the agent
anbu_review_agent = Agent(
    name="ANBU Promotion Review",
    instructions="test",
    tools=[promote_to_anbu],
)

In [3]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunResult,  # everything a finished run gives back, like final_output and last_agent
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Helper: runs one promotion request until it pauses on the tool call; reused by the demos below
async def run_until_paused() -> RunResult:
    # Script the model
    model = ScriptedModel(
        [
            [
                function_call("promote_to_anbu", {"ninja_name": "Sasuke Uchiha"}, call_id="c1"),
            ]
        ]
    )
    # Run the Agent
    return await Runner.run(
        anbu_review_agent,
        "Promote Sasuke Uchiha to ANBU.",
        run_config=RunConfig(model=model, workflow_name="e12-pause"),
    )

In [4]:
# Demo: shows the run pauses and the waiting call is listed as an interruption
async def demo_pause() -> None:
    paused = await run_until_paused()
    # proves needs_approval=True paused the run and did not call promote_to_anbu
    print("[interruptions]", len(paused.interruptions))
    for interruption in paused.interruptions:
        print(" ", interruption.tool_name, "| args:", interruption.arguments)


await demo_pause()
# Outside a notebook: asyncio.run(demo_pause())

[interruptions] 1
  promote_to_anbu | args: {"ninja_name":"Sasuke Uchiha"}


## Approve or reject, then resume

`result.to_state()` turns the stopped result into a `RunState`. Call `state.approve(interruption)`
to say yes, or `state.reject(interruption)` to say no. Then pass the `state` as the `input` of
`Runner.run`. Approve runs the tool. Reject skips it, and the model answers the "no".


In [5]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: stop one run and approve it, stop another run and reject it, then resume both
async def demo_approve_and_reject() -> None:
    # Pause the run, then say yes
    paused = await run_until_paused()
    state = paused.to_state()
    for interruption in paused.interruptions:
        state.approve(interruption)  # yes: the tool is allowed to run

    # Script the model
    model = ScriptedModel([[assistant_message("Sasuke Uchiha is now ANBU.")]])
    # Run the Agent
    approved = await Runner.run(
        anbu_review_agent,
        state,
        run_config=RunConfig(model=model, workflow_name="e12-approve"),
    )
    # proves approve runs the tool, and the run finishes with a final answer
    print("[approve]", approved.final_output)

    # Pause the run again, then say no
    paused = await run_until_paused()
    state = paused.to_state()
    for interruption in paused.interruptions:
        state.reject(interruption)  # no: the tool is skipped

    # Script the model
    model = ScriptedModel([[assistant_message("Promotion denied by the reviewer.")]])
    # Run the Agent
    rejected = await Runner.run(
        anbu_review_agent,
        state,
        run_config=RunConfig(model=model, workflow_name="e12-reject"),
    )
    # proves reject skips the tool, and the model still finishes with an answer to the "no"
    print("[reject]", rejected.final_output)


await demo_approve_and_reject()
# Outside a notebook: asyncio.run(demo_approve_and_reject())

[approve] Sasuke Uchiha is now ANBU.
[reject] Promotion denied by the reviewer.


## Save a paused run with `to_string()` and `from_string()`

In a real app, the run stops in one program and the human answers later, maybe in another program.
So the state must be saved somewhere. `state.to_string()` turns it into text, and
`RunState.from_string(agent, text)` turns the text back into a state.
Gotcha: `from_string` is `async`. If you forget `await`, you get a coroutine, not a `RunState`.
`state.add_input(...)` adds extra input, like a reviewer's note, before the run starts again.


In [6]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunState,  # a paused run that you can save, then approve or reject
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: save a stopped run as text, load it back like a new program, approve it, and resume
async def demo_serialize_and_resume() -> None:
    paused = await run_until_paused()
    # Turn the stopped run into plain text, so it can be saved in a file or database
    saved_text = paused.to_state().to_string()
    print("[saved, shortened]", saved_text[:80], "...")

    # Load the state back; await is required
    restored = await RunState.from_string(anbu_review_agent, saved_text)
    for interruption in restored.get_interruptions():
        restored.approve(interruption)
    # Add a note to the run, so the model can read why the reviewer said yes
    restored.add_input("Reviewer note: approved after checking the combat record.")

    # Script the model
    model = ScriptedModel([[assistant_message("Sasuke Uchiha is now ANBU, per reviewer approval.")]])
    # Run the Agent
    result = await Runner.run(
        anbu_review_agent,
        restored,
        run_config=RunConfig(model=model, workflow_name="e12-restore"),
    )
    # proves a saved and loaded RunState still resumes correctly, with the add_input() note
    print("[final]", result.final_output)


await demo_serialize_and_resume()
# Outside a notebook: asyncio.run(demo_serialize_and_resume())

[saved, shortened] {
  "$schemaVersion": "1.17",
  "current_turn": 1,
  "current_agent": {
    "nam ...
[final] Sasuke Uchiha is now ANBU, per reviewer approval.


## Run it live — optional, needs a real key

The whole flow with a real model. There are three live stops: one approved, one rejected, and one
saved with `to_string()`, loaded with `await RunState.from_string(...)`, given a reviewer note with
`add_input(...)`, approved, and resumed. A notebook cannot show a human clicking a button, so the
code makes each decision where a real app would wait for a reviewer.

A `promotions` list proves the tool's code only runs after a yes. It does not grow while the run is
stopped, and a "no" never adds to it. The stop is certain once the model calls the tool, because the SDK
enforces `needs_approval=True`. But whether the model calls the tool is up to the model, so each run checks for
interruptions first. Skipped when there is no `OPENAI_API_KEY`.


In [7]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


promotions: list[str] = []  # fills up only when the tool's code really runs


# Define the tool
# @function_tool callback: the same promotion action, but it writes each real run into promotions
@function_tool(name_override="promote_to_anbu", needs_approval=True)
def promote_to_anbu_live(ninja_name: str) -> str:
    """Promote a ninja to ANBU. Requires human approval."""
    promotions.append(ninja_name)
    return f"{ninja_name} has been promoted to ANBU."

In [8]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)

# Initialize the agent
live_review_agent = Agent(
    name="ANBU Promotion Review",
    instructions="When asked to promote a ninja, call promote_to_anbu, then briefly report the outcome.",
    # tools: the tools this agent is allowed to call
    tools=[promote_to_anbu_live],
)

In [9]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunResult,  # everything a finished run gives back, like final_output and last_agent
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)


# Live helper: runs one real promotion request until it stops on the tool call; used by each demo below
async def pause_live(ninja_name: str, label: str) -> RunResult | None:
    # Run the Agent
    paused = await Runner.run(
        live_review_agent,
        f"Promote {ninja_name} to ANBU.",
        run_config=RunConfig(workflow_name=f"e12-live-{label}"),
    )

    if not paused.interruptions:
        print(f"[live {label}] the model answered without calling promote_to_anbu:", paused.final_output)
        return None

    for interruption in paused.interruptions:
        # proves needs_approval=True stopped a real run, and the tool's code has not run yet
        print(
            f"[live {label} paused] {interruption.tool_name} {interruption.arguments} | promotions so far: {promotions}"
        )
    return paused

## Approve a stopped live run


In [10]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: stop a real run, say yes, and resume it
async def demo_live_approve() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    paused = await pause_live("Sasuke Uchiha", "approve")
    if paused is None:
        return

    state = paused.to_state()
    for interruption in paused.interruptions:
        state.approve(interruption)

    # Run the Agent: resume with the approved state, and now the tool really runs
    result = await Runner.run(
        live_review_agent,
        state,
        run_config=RunConfig(workflow_name="e12-live-approve-resume"),
    )

    # proves approve() let the resumed real run use the tool's code and finish
    print(f"[live approved] promotions: {promotions} | final_output: {result.final_output}")


await demo_live_approve()
# Outside a notebook: asyncio.run(demo_live_approve())

[live approve paused] promote_to_anbu {"ninja_name":"Sasuke Uchiha"} | promotions so far: []
[live approved] promotions: ['Sasuke Uchiha'] | final_output: Sasuke Uchiha has been promoted to ANBU.


## Reject a stopped live run


In [11]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: stop a real run, say no, and resume it
async def demo_live_reject() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    paused = await pause_live("Sakura Haruno", "reject")
    if paused is None:
        return

    state = paused.to_state()
    for interruption in paused.interruptions:
        state.reject(interruption)

    # Run the Agent: resume with the rejected state, and the tool never runs
    result = await Runner.run(
        live_review_agent,
        state,
        run_config=RunConfig(workflow_name="e12-live-reject-resume"),
    )

    # proves reject() stopped the tool's code (no Sakura in promotions) and the model answered the "no"
    print(f"[live rejected] promotions: {promotions} | final_output: {result.final_output}")


await demo_live_reject()
# Outside a notebook: asyncio.run(demo_live_reject())

[live reject paused] promote_to_anbu {"ninja_name":"Sakura Haruno"} | promotions so far: ['Sasuke Uchiha']
[live rejected] promotions: ['Sasuke Uchiha'] | final_output: Promotion not completed: human approval was not granted.


## Save a stopped live run, then resume it


In [12]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunState,  # a paused run that you can save, then approve or reject
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: stop a real run, save it as text, load it back like a new program, add a note, approve, and resume
async def demo_live_save_and_resume() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    paused = await pause_live("Kakashi Hatake", "serialize")
    if paused is None:
        return

    # Turn the stopped run into plain text, so it can be saved in a file or database
    saved_text = paused.to_state().to_string()
    print("[live saved, shortened]", saved_text[:80], "...")

    restored = await RunState.from_string(live_review_agent, saved_text)  # await is required
    for interruption in restored.get_interruptions():
        restored.approve(interruption)

    # Add a note to the run, so the model can read why the reviewer said yes
    restored.add_input("Reviewer note: approved after verifying combat record. Mention this note.")

    # Run the Agent: resume with the loaded, approved state plus the reviewer's note
    result = await Runner.run(
        live_review_agent,
        restored,
        run_config=RunConfig(workflow_name="e12-live-restored-resume"),
    )

    # proves a real stopped run can be saved and loaded, and resumes with the add_input() note
    print(f"[live restored] promotions: {promotions} | final_output: {result.final_output}")


await demo_live_save_and_resume()
# Outside a notebook: asyncio.run(demo_live_save_and_resume())

[live serialize paused] promote_to_anbu {"ninja_name":"Kakashi Hatake"} | promotions so far: ['Sasuke Uchiha']
[live saved, shortened] {
  "$schemaVersion": "1.17",
  "current_turn": 1,
  "current_agent": {
    "nam ...
[live restored] promotions: ['Sasuke Uchiha', 'Kakashi Hatake'] | final_output: Reviewer note: approved after verifying combat record.


## Summary

- `@function_tool(needs_approval=True)` stops the run instead of running the tool. The waiting
  call is in `RunResult.interruptions`. Use `to_state()`, then `approve()` or `reject()`, then
  `Runner.run(agent, state)` to start again.
- `to_state().to_string()` and `await RunState.from_string(...)` save a stopped run so another
  program can finish it, and `add_input(...)` adds a note before it starts again.
